In [ ]:
# resume_fft_detector.py
import os
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image, ImageFilter
import io
from sklearn.metrics import accuracy_score, roc_auc_score

# ----------------- Config (adjust if needed) -----------------
DATA_DIR = "./watermark_dataset"
CHECKPOINT_PATH = "fft_detector_resnet18_on_the_fly.pth"  # or "fft_detector_ckpt_full.pth"
BATCH_SIZE = 8
NUM_WORKERS = 0
TOTAL_NUM_EPOCHS = 50          # total epochs you want to reach (resume will continue until this)
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
VALIDATION_SPLIT = 0.15
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5
SAVE_EVERY_EPOCHS = 1          # how often to save full checkpoint
# ------------------------------------------------------------

# ---------------- Augmentations (image-space) ----------------
# Enriched train_transforms — each optional action wrapped in RandomApply
def jpeg_compress_pil(img: Image.Image, quality: int = 85):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


class RandomJPEG:
    def __init__(self, p=0.5, q_range=(60, 95)):
        self.p = p
        self.q_range = q_range

    def __call__(self, img):
        if random.random() < self.p:
            q = random.randint(self.q_range[0], self.q_range[1])
            return jpeg_compress_pil(img, q)
        return img


class RandomGaussianNoise:
    def __init__(self, p=0.5, std=0.01):
        self.p = p
        self.std = std

    def __call__(self, img):
        if random.random() < self.p:
            arr = np.array(img).astype(np.float32) / 255.0
            noise = np.random.normal(0, self.std, arr.shape).astype(np.float32)
            arr = np.clip(arr + noise, 0.0, 1.0)
            img2 = Image.fromarray((arr * 255).astype(np.uint8))
            return img2
        return img

def make_train_image_augmentations(IMAGE_SIZE):
    # Compose PIL-based augmentations (randomly applied)
    aug_list = []
    # random rotation small
    aug_list.append(
        transforms.RandomApply([transforms.RandomRotation(degrees=15)], p=0.5)
    )
    # random resized crop (sometimes)
    aug_list.append(
        transforms.RandomApply(
            [transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0))], p=0.6
        )
    )
    # horizontal flip
    aug_list.append(transforms.RandomHorizontalFlip(p=0.5))
    # color jitter
    aug_list.append(
        transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.1, 0.05)], p=0.6)
    )
    # JPEG
    aug_list.append(RandomJPEG(p=0.3, q_range=(60, 95)))
    # RandAugment (if torchvision supports it) - wrapped
    try:
        from torchvision.transforms import RandAugment

        aug_list.append(transforms.RandomApply([RandAugment()], p=0.25))
    except Exception:
        pass
    # Gaussian blur sometimes
    aug_list.append(
        transforms.RandomApply(
            [
                lambda img: img.filter(
                    ImageFilter.GaussianBlur(radius=random.uniform(0.1, 1.8))
                )
            ],
            p=0.25,
        )
    )
    # Add gaussian pixel noise sometimes
    aug_list.append(RandomGaussianNoise(p=0.25, std=0.02))
    # brightness jitter more finely (RandomApply)
    aug_list.append(
        transforms.RandomApply([transforms.ColorJitter(brightness=(0.8, 1.2))], p=0.5)
    )

    # final: ensure image is resized to IMAGE_SIZE (if not already)
    final = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), *aug_list])
    return final

IMAGE_SIZE = 512  # might adjust to your pipeline / VAE size
IMG_AUG = make_train_image_augmentations(IMAGE_SIZE)
TEST_AUG = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ]
)

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ----------------- assume helpers are available from your notebook -----------------
def transform_img(image, target_size=512):
    tform = transforms.Compose(
        [
            transforms.Resize(target_size),
            transforms.CenterCrop(target_size),
            transforms.ToTensor(),
            transforms.ConvertImageDtype(torch.float32),
        ]
    )
    image = tform(image)
    return 2.0 * image - 1.0



# ---------------- Data loader that does augment -> pipe -> forward_diffusion -> FFT ----------------
class WatermarkOnTheFlyDataset(Dataset):
    """
    Loads image files and labels, applies augmentations, then runs:
      tsr_img -> pipe.get_image_latents(sample=False) -> pipe.forward_diffusion(...) -> FFT
    Returns: (fft_channels_tensor (float32, shape (2*C, H, W)), label)
    """

    def __init__(
        self,
        file_paths,
        labels,
        pipe,
        text_embeddings,
        num_inference_steps,
        guidance_scale=1.0,
        device="cpu",
        image_aug=IMG_AUG,
    ):
        assert len(file_paths) == len(labels)
        self.file_paths = file_paths
        self.labels = labels
        self.pipe = pipe
        self.text_embeddings = text_embeddings
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.device = device
        self.image_aug = image_aug
        self.return_reversed_latents = False  # default behavior

    def __len__(self):
        return len(self.file_paths)

    def _load_pil(self, fp):
        # accept PIL.Image, numpy array, or path
        if isinstance(fp, Image.Image):
            return fp.convert("RGB")
        if isinstance(fp, torch.Tensor):
            # convert tensor (C,H,W) to PIL
            arr = (fp.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            return Image.fromarray(arr)
        p = Path(fp)
        img = Image.open(p).convert("RGB")
        return img
    
    def set_return_reversed_latents(self, return_reversed_latents: bool):
        self.return_reversed_latents = return_reversed_latents

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = int(self.labels[idx])

        pil_img = self._load_pil(path)

        # --- AUGMENT IMAGE FIRST ---
        img_aug = self.image_aug(pil_img)

        # convert to tensor and to device/dtype for pipe
        tsr_img = transform_img(img_aug).unsqueeze(
            0
        )  # (1,C,H,W)
        # move to correct dtype & device for the unet/vae as the user did earlier:
        target_dtype = next(self.pipe.unet.parameters()).dtype
        tsr_img = tsr_img.to(dtype=target_dtype, device=self.device)

        # --- encode to image latents ---
        with torch.no_grad():
            image_latents = self.pipe.get_image_latents(
                tsr_img, sample=False
            )  # user's helper expects (C,H,W) or (B,C,H,W)
            # ensure batch dim
            if image_latents.ndim == 3:
                image_latents = image_latents.unsqueeze(0)

            
            # --- forward/inversion -> x_T (depending on forward_diffusion implementation)
            reversed_latents = self.pipe.forward_diffusion(
                latents=image_latents,
                text_embeddings=self.text_embeddings,
                guidance_scale=1,
                num_inference_steps=self.num_inference_steps,
            )  # expect tensor shape (B,C,H,W)

            if self.return_reversed_latents:
                return reversed_latents[0], torch.tensor(label, dtype=torch.long)

            # Keep complex-safe: cast to float32 after splitting real/imag
            # Compute FFT (complex)
            vis_latent_fft = torch.fft.fftshift(
                torch.fft.fft2(reversed_latents), dim=(-1, -2)
            )  # (B,C,H,W) complex
            # we assume batch==1
            fft_b = vis_latent_fft[0]

            assert fft_b.ndim == 3  # (C,H,W) complex

            # convert to float channels (real, imag) as float32
            real = fft_b.real.to(dtype=torch.float32)
            imag = fft_b.imag.to(dtype=torch.float32)
            fft_ch = torch.cat([real, imag], dim=0)  # (2*C, H, W)

        return fft_ch, torch.tensor(label, dtype=torch.long)


# ---------------- helper to collect files and labels ----------------
def discover_dataset_files(data_dir: str):
    """
    Discover files in data_dir. Support:
      - subfolders 'watermarked' and 'clean' (or any two subfolders)
      - .pt files with keys
    Returns lists: file_paths, labels
    """
    p = Path(data_dir)
    if not p.exists():
        raise RuntimeError(f"{data_dir} not found")

    # case: two subfolders inside (binary classes)
    subdirs = [d for d in p.iterdir() if d.is_dir()]
    if len(subdirs) >= 2:
        # choose the first two directories as classes
        classes = sorted(subdirs)[:2]
        file_paths = []
        labels = []
        for label, cdir in enumerate(classes):
            exts = list(cdir.glob("*"))
            for f in exts:
                if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".pt", ".pth"]:
                    file_paths.append(str(f))
                    labels.append(label)
        return file_paths, labels

    # case: many .pt files with 'fft' or 'image' and 'label'
    pts = list(p.glob("*.pt"))
    if len(pts) > 0:
        file_paths = []
        labels = []
        for f in pts:
            try:
                d = torch.load(f)
                if isinstance(d, dict) and "label" in d:
                    file_paths.append(str(f))
                    labels.append(int(d["label"]))
            except Exception:
                continue
        if len(file_paths) > 0:
            return file_paths, labels

    # fallback: collect images in folder and try to infer labels by filename (contains 'water' or 'wm')
    imgs = [
        str(f)
        for f in p.glob("*")
        if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]
    ]
    if len(imgs) > 0:
        file_paths = []
        labels = []
        for f in imgs:
            fname = os.path.basename(f).lower()
            lbl = 0
            if "water" in fname or "wm" in fname or "marked" in fname or "1_" in fname:
                lbl = 1
            file_paths.append(f)
            labels.append(lbl)
        return file_paths, labels

    raise RuntimeError(
        "Unable to discover dataset files. Please structure dataset as subfolders or .pt files with label key."
    )


# ---------------- small model helper ----------------
def make_model(in_channels):
    model = models.resnet18(pretrained=True)
    # adapt first conv
    model.conv1 = nn.Conv2d(
        in_channels,
        model.conv1.out_channels,
        kernel_size=model.conv1.kernel_size,
        stride=model.conv1.stride,
        padding=model.conv1.padding,
        bias=(model.conv1.bias is not None),
    )
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


# ---------------- training / eval loops ----------------
def train_epoch(model, loader, opt, crit):
    model.train()
    running_loss = 0.0
    for X, y in tqdm(loader, desc="train", leave=False):
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        opt.zero_grad()
        out = model(X)
        loss = crit(out, y)
        loss.backward()
        opt.step()
        running_loss += loss.item() * X.size(0)
    return running_loss / len(loader.dataset)

def eval_model(model, loader):
    model.eval()
    trues, preds, probs = [], [], []
    running_loss = 0.0
    with torch.no_grad():
        for X, y in tqdm(loader, desc="eval", leave=False):
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            print(X.shape)
            out = model(X)
            loss = crit(out, y)
            p = torch.softmax(out, dim=1)[:, 1].detach().cpu().numpy()
            pred = (p > 0.5).astype(int).tolist()
            probs.extend(p.tolist())
            preds.extend(pred)
            trues.extend(y.cpu().numpy().tolist())
            running_loss += loss.item() * X.size(0)

    acc = accuracy_score(trues, preds)
    try:
        auc = roc_auc_score(trues, probs)
    except Exception:
        auc = float("nan")
    return acc, auc, (running_loss / len(loader.dataset))
# -----------------------------------------------------------------------------------

# Validate that PIPE and TEXT_EMBEDDINGS are present (or load them here)
try:
    import torch
    import diffusers
    from diffusers import DPMSolverMultistepScheduler
    from inverse_stable_diffusion import InversableStableDiffusionPipeline

    model_id = "stabilityai/stable-diffusion-2-1-base"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    scheduler = DPMSolverMultistepScheduler.from_pretrained(model_id, subfolder="scheduler")
    pipe = InversableStableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch.float16,
        revision="fp16",
        verbose=False,
    )
    diffusers.utils.logging.disable_progress_bar()
    pipe.set_progress_bar_config(disable=True)
    pipe = pipe.to(device)

    TEXT_EMBEDDINGS = pipe.get_text_embedding("")  #
    PIPE = pipe  # make sure 'pipe' is in scope
# ------------------------------------------
except NameError:
    raise RuntimeError("Please ensure `PIPE` and `TEXT_EMBEDDINGS` are available in the runtime before running resume script.")

# discover files
file_paths, labels = discover_dataset_files(DATA_DIR)
combined = list(zip(file_paths, labels))
random.shuffle(combined)
file_paths, labels = zip(*combined)
n_val = int(len(file_paths) * VALIDATION_SPLIT)
val_paths = file_paths[:n_val]
val_labels = labels[:n_val]
train_paths = file_paths[n_val:]
train_labels = labels[n_val:]

# create datasets (use same constructor as before)
train_ds = WatermarkOnTheFlyDataset(
    train_paths,
    train_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
)

val_ds = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=IMG_AUG
    
)

# infer input channels from a sample (may be expensive, but needed to create model)
sample_fft, _ = train_ds[0]
in_ch = sample_fft.shape[0]

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


# build model + optimizer + criterion
model = make_model(in_ch).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()

# try to load checkpoint
if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    # two possible styles:
    # 1) legacy single state_dict saved by torch.save(model.state_dict())
    # 2) full checkpoint dict saved with model_state_dict, optimizer_state_dict, epoch, maybe scheduler
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            try:
                opt.load_state_dict(ckpt["optimizer_state_dict"])
            except Exception as e:
                print("Warning: couldn't fully load optimizer state:", e)
        if "epoch" in ckpt:
            start_epoch = int(ckpt["epoch"]) + 1
        print(f"Restored model and optimizer. Resuming from epoch {start_epoch}")
    else:
        # assume ckpt is a plain state_dict
        try:
            model.load_state_dict(ckpt)
            print("Loaded model.state_dict() from checkpoint (optimizer state not present).")
        except Exception as e:
            raise RuntimeError("Checkpoint format not recognized and failed to load model:", e)
else:
    print("No checkpoint found; training from scratch.")

# optional: a scheduler - if you used one earlier and saved state, restore here
# scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
# if 'scheduler_state_dict' in ckpt: scheduler.load_state_dict(ckpt['scheduler_state_dict'])

evaluate_times = 5
# all_accs, all_aucs, all_val_losses = [], [], []
# for e_i in range(evaluate_times):
#     acc, auc, val_loss = eval_model(model, val_loader)
#     all_accs.append(acc)
#     all_aucs.append(auc)
#     all_val_losses.append(val_loss)
#     print(f"  val acc: {acc:.4f}  val AUROC: {auc:.4f}  val_loss: {val_loss:.4f}")
# print("Training complete.")


c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\diffusers\pipelines\pipeline_loading_utils.py:333: FutureWarning: You are loading the variant fp16 from stabilityai/stable-diffusion-2-1-base via `revision='fp16'`. This behavior is deprecated and will be removed in diffusers v1. One should use `variant='fp16'` instead. However, it appears that stabilityai/stable-diffusion-2-1-base currently does not have the required variant filenames in the 'main' branch. 
 The Diffusers team and community would be very grateful if you could open an issue: https://github.com/huggingface/diffusers/issues/new with the title 'stabilityai/stable-diffusion-2-1-base is missing fp16 files' so that the correct variant file can be added.
  warnin

Loading checkpoint: fft_detector_resnet18_on_the_fly.pth
Loaded model.state_dict() from checkpoint (optimizer state not present).


C:\Users\mike8\AppData\Local\Temp\ipykernel_783104\830923924.py:233: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\EmptyTensor.cpp:58.)
  torch.fft.fft2(reversed_latents), dim=(-1, -2)
c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [2]:
# On Augmented dataset, the performance is:
# print(f"ACC [{np.array(all_accs).mean():.4f}+-{np.array(all_accs).std():.4f}] | AUROC [{np.array(all_aucs).mean():.4f}+-{np.array(all_aucs).std():.4f}] |")
# ACC [0.7360+-0.0172] | AUROC [0.8077+-0.0146] |

In [ ]:
val_ds_no_aug = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=TEST_AUG,
)
val_loader_no_aug = DataLoader(val_ds_no_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


all_accs, all_aucs, all_val_losses = [], [], []
for e_i in range(1):
    acc, auc, val_loss = eval_model(model, val_loader_no_aug)
    all_accs.append(acc)
    all_aucs.append(auc)
    all_val_losses.append(val_loss)
    print(f"  val acc: {acc:.4f}  val AUROC: {auc:.4f}  val_loss: {val_loss:.4f}")
print("Training complete.")


In [4]:
# On Not Augmented dataset, the performance is:
# print(f"ACC [{np.array(all_accs).mean():.4f}+-{np.array(all_accs).std():.4f}] | AUROC [{np.array(all_aucs).mean():.4f}+-{np.array(all_aucs).std():.4f}] |")
# ACC [0.8400+-0.0000] | AUROC [0.9802+-0.0000] |

In [5]:
import matplotlib.pyplot as plt

try:
    from scipy.stats import ncx2
except Exception as e:
    ncx2 = None
    _scipy_import_error = e

# requires: numpy, torch, matplotlib (for plotting), scipy
import numpy as np
import torch
try:
    from scipy.stats import ncx2
except Exception as e:
    ncx2 = None
    _scipy_import_error = e

def _compute_psnr_masked(pred, target, mask, eps=1e-8):
    """
    Fallback PSNR on masked region: 10*log10(MAX^2 / MSE_masked).
    Assumes inputs in [0,1] or any bounded range; uses dynamic MAX from target.
    """
    # restrict to mask
    diff = (pred - target)[mask]
    mse = (diff.float() ** 2).mean().clamp_min(eps)
    # dynamic range from target over mask
    t = target[mask].float()
    max_val = (t.max() - t.min()).clamp_min(eps)
    # if target is constant, fall back to 1.0 range
    if max_val.item() == 0.0:
        max_val = torch.tensor(1.0, device=t.device)
    psnr = 10.0 * torch.log10((max_val ** 2) / mse)
    return psnr.item()

def eval_watermark(
    latent: torch.Tensor,
    watermarking_mask: torch.Tensor,
    gt_patch: torch.Tensor,
    w_measurement: str,
) -> float:
    """
    Evaluate watermark quality for a SINGLE latent against gt_patch on watermarking_mask.

    Args:
        latent:              Tensor [..., H, W] or [..., C, H, W] (last two dims are H,W).
        watermarking_mask:   Bool/byte mask broadcastable to latent/gt_patch spatial dims.
        gt_patch:            Ground-truth pattern tensor, same shape as latent (or broadcastable).
        w_measurement:       e.g. "complex+l1", "seed+l1", "complex+psnr", "seed+psnr".

    Returns:
        metric: float scalar.
    """
    # choose domain
    if "complex" in w_measurement:
        # FFT over spatial dims (H, W); shift to center
        latent_dom = torch.fft.fftshift(torch.fft.fft2(latent), dim=(-1, -2))
        target_dom = gt_patch
    elif "seed" in w_measurement:
        # real domain
        latent_dom = latent
        target_dom = gt_patch
    else:
        raise NotImplementedError(f"w_measurement domain not recognized: {w_measurement}")

    # ensure mask is boolean and broadcastable to spatial dims
    wm_mask = watermarking_mask.bool()

    # metric
    if "l1" in w_measurement:
        metric = torch.abs(latent_dom[wm_mask] - target_dom[wm_mask]).mean().item()
    elif "psnr" in w_measurement:
        # try user's compute_psnr first (signature: (pred, target, mask))
        if "compute_psnr" in globals() and callable(globals()["compute_psnr"]):
            metric = globals()["compute_psnr"](latent_dom, target_dom, wm_mask)
        else:
            metric = _compute_psnr_masked(latent_dom, target_dom, wm_mask)
    else:
        raise NotImplementedError(f"w_measurement metric not recognized: {w_measurement}")
    return metric

def _ensure_mask_shape(mask, ref_tensor):
    if not torch.is_tensor(mask):
        mask = torch.tensor(mask)
    mask = mask.bool()
    if mask.ndim == 2:
        mask = mask.unsqueeze(0).unsqueeze(0)  # 1,1,H,W
    if mask.shape != ref_tensor.shape:
        mask = mask.expand(ref_tensor.shape)
    return mask

def detect_watermark_ncx2_from_latent(
    latent_tensor,            # torch tensor (B,C,H,W)
    gt_patch,                 # key (either in freq domain or spatial)
    watermarking_mask,        # boolean mask selecting frequency bins M (H,W or B,C,H,W)
    k_in_freq_domain=True,
    alpha=0.01,
    eps=1e-12,
    verbose=True,
):
    
    l1_metric = eval_watermark(latent_tensor, watermarking_mask, gt_patch, w_measurement="l1_complex")
    psnr_metric = eval_watermark(latent_tensor, watermarking_mask, gt_patch, w_measurement="psnr_complex")

    if ncx2 is None:
        raise RuntimeError("scipy required for noncentral chi2. Import error: %s" % (_scipy_import_error,))

    # Ensure batch dim
    if latent_tensor.ndim == 3:
        latent_tensor = latent_tensor.unsqueeze(0)
    B, C, H, W = latent_tensor.shape
    if B != 1 and verbose:
        print("Warning: only processing first sample of batch")
        latent_tensor = latent_tensor[0:1]

    device = latent_tensor.device

    # compute y_fft in standard convention (fft2 + fftshift)
    # ensure we do FFT on float32 inputs so output complex dtype becomes complex64
    y_input = latent_tensor
    if y_input.dtype == torch.float16 or y_input.dtype == torch.bfloat16:
        y_input = y_input.to(torch.float32)
    y_fft = torch.fft.fftshift(torch.fft.fft2(y_input), dim=(-1, -2))  # complex dtype (likely complex64)

    # prepare key in Fourier domain
    if k_in_freq_domain:
        k_fft = gt_patch
        # If user passed a float tensor rather than complex, convert to fft
        if not torch.is_complex(k_fft):
            if k_fft.ndim == 3:
                k_fft = k_fft.unsqueeze(0)
            k_tmp = k_fft
            if k_tmp.dtype == torch.float16 or k_tmp.dtype == torch.bfloat16:
                k_tmp = k_tmp.to(torch.float32)
            k_fft = torch.fft.fftshift(torch.fft.fft2(k_tmp), dim=(-1, -2))
    else:
        k_tmp = gt_patch
        if k_tmp.ndim == 3:
            k_tmp = k_tmp.unsqueeze(0)
        if k_tmp.dtype == torch.float16 or k_tmp.dtype == torch.bfloat16:
            k_tmp = k_tmp.to(torch.float32)
        k_fft = torch.fft.fftshift(torch.fft.fft2(k_tmp), dim=(-1, -2))

    # ensure shapes align (replicate channels if necessary)
    if k_fft.shape != y_fft.shape:
        if k_fft.shape[0] == 1 and y_fft.shape[0] == 1 and k_fft.shape[1] == 1 and y_fft.shape[1] > 1:
            k_fft = k_fft.repeat(1, y_fft.shape[1], 1, 1)
        else:
            raise ValueError(f"Shape mismatch y_fft {y_fft.shape} vs k_fft {k_fft.shape}")

    # Convert complex-half types to complex64 before moving to CPU / numpy
    if y_fft.is_complex() and y_fft.dtype != torch.complex64 and y_fft.dtype != torch.complex128:
        # prefer complex64
        y_fft = y_fft.to(torch.complex64)
    if k_fft.is_complex() and k_fft.dtype != torch.complex64 and k_fft.dtype != torch.complex128:
        k_fft = k_fft.to(torch.complex64)

    # prepare mask on CPU for indexing
    mask = _ensure_mask_shape(watermarking_mask, y_fft)  # boolean on same shape
    mask_cpu = mask.detach().cpu()

    # Move fft tensors to CPU (complex64) for numpy/scipy interaction
    y_cpu = y_fft.detach().cpu()
    k_cpu = k_fft.detach().cpu()

    # select masked elements and flatten
    y_sel = y_cpu[mask_cpu].numpy().astype(np.complex128).ravel()
    k_sel = k_cpu[mask_cpu].numpy().astype(np.complex128).ravel()

    M = y_sel.size
    if M == 0:
        raise ValueError("Empty watermark mask (|M|=0)")

    # compute sigma2, eta, lam in numpy (scalars)
    sigma2 = float(np.mean(np.abs(y_sel)**2))
    sigma2 = max(sigma2, eps)
    eta = (1.0 / sigma2) * float(np.sum(np.abs(k_sel - y_sel)**2))
    lam = (1.0 / sigma2) * float(np.sum(np.abs(k_sel)**2))

    # p-value and threshold via non-central chi2
    p_value = float(ncx2.cdf(eta, df=M, nc=lam))
    try:
        threshold_eta = float(ncx2.ppf(alpha, df=M, nc=lam))
    except Exception:
        threshold_eta = None

    detected = (p_value <= alpha)

    if verbose:
        print(f"ncx2 test: M={M}, sigma2={sigma2:.4e}, eta={eta:.4f}, lam={lam:.4f}, p={p_value:.4e}, alpha={alpha}, detected={detected}")

    return dict(
        p_value=p_value,
        eta=eta,
        threshold_eta=threshold_eta,
        detected=bool(detected),
        sigma2=sigma2,
        M=int(M),
        lam=lam,
        alpha=float(alpha),
        l1_metric=float(l1_metric),
        psnr_metric=float(psnr_metric),
    )

# ---------------------------
# Full integrated verify function
# ---------------------------
def verify_with_ncx2(
    image,
    pipe,
    text_embeddings,
    watermarking_mask,
    gt_patch,
    num_inference_steps,
    alpha=0.01,
    k_in_freq_domain=True,
    show_visual=True,
    device=None,
):
    """
    Integrates the paper's ncx2 test into your verify routine.
    Parameters:
      - image: PIL or np array as before
      - pipe: your diffusion pipeline with get_image_latents and forward_diffusion
      - text_embeddings: embeddings used in forward_diffusion call (if needed)
      - watermarking_mask: boolean mask in frequency domain (H,W) or (1,C,H,W)
      - gt_patch: your key (if k_in_freq_domain True, gt_patch must already be in FFT domain or same shape)
      - num_inference_steps: used when calling forward_diffusion (your pipeline)
      - alpha: significance level (paper uses 0.01)
      - k_in_freq_domain: whether gt_patch is already Fourier (True by default)
    Returns:
      - results dict with detection & metrics
    """
    if device is None:
        device = next(pipe.unet.parameters()).device if hasattr(pipe, "unet") else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # show image
    if show_visual:
        plt.figure(figsize=(4,4))
        plt.imshow(image)
        plt.axis("off")
        plt.show()

    # to tensor and move to dtype/device expected by pipeline
    tsr_img = transform_img(image).unsqueeze(0).to(next(pipe.unet.parameters()).dtype).to(device)
    print(tsr_img.shape, tsr_img.dtype)  # torch.Size([1, 3, 512, 512]) torch.float16
    # encode to image latents (user's helper)
    image_latents = pipe.get_image_latents(tsr_img, sample=False)
    print(image_latents.shape, image_latents.dtype) # torch.Size([1, 4, 64, 64]) torch.float16

    reversed_latents = pipe.forward_diffusion(
        latents=image_latents,
        text_embeddings=text_embeddings,
        guidance_scale=1,
        num_inference_steps=num_inference_steps,
    )

    # Visualise in Fourier domain (keep as complex)
    vis_latent_fft = torch.fft.fftshift(torch.fft.fft2(reversed_latents), dim=(-1,-2))
    
    if show_visual:
        # display magnitude for first channel
        mag = vis_latent_fft[0,0].abs().detach().cpu().numpy()
        plt.figure(figsize=(4,4))
        plt.title("FFT magnitude (reversed_latents)")
        plt.imshow(np.log1p(mag), cmap="magma")
        plt.axis("off")
        plt.show()

    # compute l1/psnr metrics if you still want them (assuming eval_watermark accepts complex tensors)
    l1_metric = None
    psnr_metric = None
    try:
        l1_metric = eval_watermark(reversed_latents, watermarking_mask, gt_patch, w_measurement="l1_complex")
        psnr_metric = eval_watermark(reversed_latents, watermarking_mask, gt_patch, w_measurement="psnr_complex")
    except Exception as e:
        # not fatal — just print
        print("Warning: eval_watermark failed:", e)

    # run statistical test (paper)
    stats = detect_watermark_ncx2_from_latent(
        latent_tensor=reversed_latents,
        gt_patch=gt_patch,
        watermarking_mask=watermarking_mask,
        k_in_freq_domain=k_in_freq_domain,
        alpha=alpha,
        verbose=True,
    )

    # nicely return all info
    results = dict(
        l1_metric=l1_metric,
        psnr_metric=psnr_metric,
        ncx2_stats=stats,
    )
    return results

In [6]:
# Watermarking parameters
w_seed = 999999
w_channel = 0
w_pattern = 'logpolar_grid' 
# w_pattern = 'ring' 
w_mask_shape = 'circle'
w_radius = 8
# w_measurement = 'l1_complex'
w_measurement = 'psnr_complex'
w_injection = 'complex'
w_pattern_const  = 0
w_strength = 0.9
num_inference_steps = 50


In [7]:
from utils.random import set_random_seed
import numpy as np
import copy

import warnings

warnings.filterwarnings("ignore", category=UserWarning, message=".*?.*?")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*?.*?")

def circle_mask(size=64, r=10, x_offset=0, y_offset=0):
    """
    Create a circular mask of given size and radius.
    Parameters:
        - size: The size of the mask (width and height).
        - r: The radius of the circle.
        - x_offset: The x-coordinate offset for the center of the circle.
        - y_offset: The y-coordinate offset for the center of the circle.
    Returns:
        - mask: A boolean mask where True values represent the circular area.

    Reference: https://stackoverflow.com/questions/69687798/generating-a-soft-circluar-mask-using-numpy-python-3
    """
    x0 = y0 = size // 2
    x0 += x_offset
    y0 += y_offset
    y, x = np.ogrid[:size, :size]
    y = y[::-1]
    return ((x - x0) ** 2 + (y - y0) ** 2) <= r**2


import math
import copy
import torch
import torch.nn.functional as F
import numpy as np

# ------------------------
# Helpers
# ------------------------


def circle_mask(size=64, r=10, x_offset=0, y_offset=0):
    """Return a boolean NumPy circular mask (size x size) with radius r."""
    x0 = y0 = size // 2
    x0 += x_offset
    y0 += y_offset
    y, x = np.ogrid[:size, :size]
    y = y[::-1]
    return ((x - x0) ** 2 + (y - y0) ** 2) <= r**2


def _ensure_bchw(x, device=None, dtype=None):
    if not torch.is_tensor(x):
        raise TypeError("expected torch.Tensor")
    if x.ndim == 2:
        x = x.unsqueeze(0).unsqueeze(0)
    elif x.ndim == 3:
        x = x.unsqueeze(0)
    if device is not None:
        x = x.to(device)
    if dtype is not None:
        x = x.to(dtype)
    return x


def cartesian_to_logpolar(img, H_lp=None, W_lp=None, r_min=1.0, eps=1e-6):
    # img: (B,C,H,W)
    B, C, H, W = img.shape
    device = img.device
    dtype = img.dtype

    if H_lp is None:
        H_lp = H
    if W_lp is None:
        W_lp = W

    cx = (W - 1) / 2.0
    cy = (H - 1) / 2.0
    R_max = math.hypot(cx, cy)

    log_r_min = math.log(r_min + eps)
    log_r_max = math.log(R_max + eps)

    # radius samples (H_lp), use torch.linspace with positional steps
    r_lin = torch.exp(
        torch.linspace(log_r_min, log_r_max, H_lp, device=device, dtype=dtype)
    )
    # theta samples (W_lp) - use positional steps and endpoint=False
    theta_lin = torch.linspace(
        0.0, 2.0 * math.pi, W_lp, device=device, dtype=dtype, endpoint=False
    )

    r_grid, theta_grid = torch.meshgrid(r_lin, theta_lin, indexing="ij")  # (H_lp, W_lp)

    x = cx + r_grid * torch.cos(theta_grid)
    y = cy + r_grid * torch.sin(theta_grid)

    x_norm = (x / (W - 1)) * 2.0 - 1.0
    y_norm = (y / (H - 1)) * 2.0 - 1.0

    grid = torch.stack((x_norm, y_norm), dim=-1)  # (H_lp, W_lp, 2)
    grid = grid.unsqueeze(0).expand(B, -1, -1, -1)  # (B, H_lp, W_lp, 2)

    logpolar = F.grid_sample(
        img, grid, mode="bilinear", padding_mode="zeros", align_corners=True
    )
    return logpolar


def logpolar_to_cartesian(lp_img, out_H, out_W, r_min=1.0, eps=1e-6):
    B, C, H_lp, W_lp = lp_img.shape
    device = lp_img.device
    dtype = lp_img.dtype

    cx = (out_W - 1) / 2.0
    cy = (out_H - 1) / 2.0
    R_max = math.hypot(cx, cy)

    log_r_min = math.log(r_min + eps)
    log_r_max = math.log(R_max + eps)

    xs = torch.linspace(0.0, out_W - 1, out_W, device=device, dtype=dtype)
    ys = torch.linspace(0.0, out_H - 1, out_H, device=device, dtype=dtype)
    y_grid, x_grid = torch.meshgrid(ys, xs, indexing="ij")  # (out_H, out_W)

    dx = x_grid - cx
    dy = y_grid - cy
    r = torch.sqrt(dx * dx + dy * dy)
    theta = torch.atan2(dy, dx)
    theta = torch.where(theta < 0.0, theta + 2.0 * math.pi, theta)

    r_clamped = torch.clamp(r, min=1.0, max=R_max)
    log_r = torch.log(r_clamped + eps)
    u = (log_r - log_r_min) / (log_r_max - log_r_min)
    u = u.clamp(0.0, 1.0) * (H_lp - 1)

    v = (theta / (2.0 * math.pi)) * (W_lp - 1)

    x_src = (v / (W_lp - 1)) * 2.0 - 1.0
    y_src = (u / (H_lp - 1)) * 2.0 - 1.0

    grid = torch.stack((x_src, y_src), dim=-1).unsqueeze(0).expand(B, -1, -1, -1)
    cart = F.grid_sample(
        lp_img, grid, mode="bilinear", padding_mode="zeros", align_corners=True
    )
    return cart


def generate_logpolar_grid(
    H_lp, W_lp, n_spokes=16, n_rings=12, device="cpu", dtype=torch.float16
):
    """
    Create a synthetic log-polar canvas that contains both concentric rings and radial spokes.
    Output shape: (1,1,H_lp,W_lp) - single-channel; you can expand to match channels.
    """
    # normalized coords u in [0,1] for radius index, v in [0,1) for theta
    u = torch.linspace(0.0, 1.0, H_lp, device=device, dtype=dtype).view(H_lp, 1)
    v = torch.linspace(0.0, 1.0, W_lp, device=device, dtype=dtype).view(1, W_lp)

    # ring pattern: sinusoidal in log-radius (higher frequency near center if desired)
    rings = 0.5 * (1.0 + torch.sign(torch.sin(u * n_rings * math.pi)))  # coarse rings
    # alternative smoother rings (uncomment if you prefer smooth):
    # rings = 0.5 * (1.0 + torch.sin(u * n_rings * math.pi))

    # spoke pattern: sinusoidal in angle
    spokes = 0.5 * (1.0 + torch.sin(v * n_spokes * 2.0 * math.pi))

    # combine them multiplicatively (or additively) to produce grid intersections
    lp = rings * spokes  # shape (H_lp, W_lp)
    lp = lp.unsqueeze(0).unsqueeze(0)  # (1,1,H_lp,W_lp)
    return lp.to(device=device, dtype=dtype)


# -------------------------
# Updated get_watermarking_pattern
# -------------------------
def get_watermarking_pattern(
    pipe,
    w_seed: int,
    w_pattern: str,
    w_radius: int,
    device: torch.device,
    shape: tuple | None = None,
    strength: float = 0.9,
):
    def set_random_seed(s):
        torch.manual_seed(int(s) & 0xFFFFFFFF)

    set_random_seed(w_seed)

    # prepare initial tensor
    if shape is not None:
        gt_init = torch.randn(*shape, device=device)
    else:
        if hasattr(pipe, "get_random_latents"):
            gt_init = pipe.get_random_latents()
        else:
            if hasattr(pipe, "unet") and hasattr(pipe.unet.config, "sample_size"):
                sample_size = pipe.unet.config.sample_size
                if isinstance(sample_size, int):
                    H = W = sample_size
                else:
                    H, W = (
                        sample_size
                        if len(sample_size) >= 2
                        else (sample_size[0], sample_size[0])
                    )
                in_ch = pipe.unet.config.in_channels
                gt_init = torch.randn(1, in_ch, H, W, device=device)
            else:
                gt_init = torch.randn(1, 3, 256, 256, device=device)

    gt_init = _ensure_bchw(gt_init, device=device, dtype=torch.float32)

    # handle patterns that require FFT on float32: cast to float32 then cast back
    def safe_fft2_shift(x):
        orig_dtype = x.dtype
        x_f = x.to(torch.float32)
        fft = torch.fft.fft2(x_f)
        fft_s = torch.fft.fftshift(fft, dim=(-1, -2))
        # return cast back to original dtype (complex->real not appropriate); here we keep real part if needed
        # but most usage expects complex spectrum; if you only use magnitude/phase you should adapt accordingly.
        return fft_s.to(orig_dtype) if orig_dtype.is_floating_point else fft_s

    if "logpolar_grid" in w_pattern:
        # create synthetic LP canvas (single-channel) and expand to channels
        B, C, H, W = gt_init.shape
        H_lp = H
        W_lp = W
        # tune these counts to get desired spokes/rings
        n_spokes = max(8, int(w_radius))  # e.g. control by w_radius
        n_rings = max(6, int(w_radius // 2))
        lp_canvas = generate_logpolar_grid(
            H_lp,
            W_lp,
            n_spokes=n_spokes,
            n_rings=n_rings,
            device=device,
            dtype=gt_init.dtype,
        )

        # fft_s = safe_fft2_shift(gt_init)
        # gt_temp = fft_s.real if torch.is_complex(fft_s) else fft_s
        gt_temp = torch.fft.fftshift(torch.fft.fft2(gt_init), dim=(-1, -2))
        
        # expand channel-wise and add small noise so it's less "perfect"
        lp_canvas = lp_canvas.expand(B, C, -1, -1)
        # map back to cartesian
        gt_patch = logpolar_to_cartesian(lp_canvas, out_H=H, out_W=W)
        # normalise to mean ~0, std ~1 (optional) or to model range
        # here we scale to similar stats as gt_init
        gt_patch = (gt_patch - gt_patch.mean()) / (gt_patch.std(unbiased=False) + 1e-8)
        gt_patch = (gt_patch * strength) + (gt_temp * (1 - strength)) # scale/downweight
        
        return gt_patch.to(dtype=torch.complex32)

    # keep previous behaviors but fix dtype issues and linspace earlier (we already fixed linspace above)
    if "seed_ring" in w_pattern:
        gt_patch = gt_init
        gt_patch_tmp = copy.deepcopy(gt_patch)
        H = gt_init.shape[-1]
        for i in range(w_radius, 0, -1):
            tmp_mask = circle_mask(gt_init.shape[-1], r=i)
            tmp_mask = torch.tensor(tmp_mask, device=device, dtype=torch.bool)
            for j in range(gt_patch.shape[1]):
                gt_patch[:, j, tmp_mask] = gt_patch_tmp[0, j, 0, i].item()

    elif "seed_zeros" in w_pattern:
        gt_patch = gt_init * 0

    elif "seed_rand" in w_pattern:
        gt_patch = gt_init

    elif "rand" in w_pattern:
        # perform safe fft on float32 to avoid half dtype error
        fft_s = safe_fft2_shift(gt_init)
        # use magnitude or real-part as desired - here take real part
        gt_patch = fft_s.real if torch.is_complex(fft_s) else fft_s
        gt_patch = gt_patch * 0  # your original wanted some behaviour; adapt as needed
        gt_patch[:] = gt_patch[0]

    elif "zeros" in w_pattern:
        fft_s = safe_fft2_shift(gt_init)
        gt_patch = (fft_s.real if torch.is_complex(fft_s) else fft_s) * 0

    elif "const" in w_pattern:
        w_pattern_const = globals().get("w_pattern_const", 0.0)
        fft_s = safe_fft2_shift(gt_init)
        gt_patch = (fft_s.real if torch.is_complex(fft_s) else fft_s) * 0
        gt_patch = gt_patch + float(w_pattern_const)

    elif "ring" in w_pattern:
        gt_patch = torch.fft.fftshift(torch.fft.fft2(gt_init), dim=(-1, -2))
        gt_patch_tmp = copy.deepcopy(gt_patch)
        for i in range(w_radius, 0, -1):
            tmp_mask = circle_mask(gt_init.shape[-1], r=i)
            tmp_mask = torch.tensor(tmp_mask).to(device)

            for j in range(gt_patch.shape[1]):
                gt_patch[:, j, tmp_mask] = gt_patch_tmp[0, j, 0, i].item()

    else:
        gt_patch = gt_init

    return gt_patch.to(dtype=torch.complex32)

gt_patch = get_watermarking_pattern(
    pipe,
    w_seed=w_seed,
    w_pattern=w_pattern,
    w_radius=w_radius,
    device=device,
    strength=w_strength,
    shape=None,
)


In [8]:
def get_watermarking_mask(
    init_latents_w,
    w_mask_shape: str,
    w_channel: int,
    w_radius: float,
    device: str,
):
    """
    Generate a watermarking mask based on the specified parameters.
    Parameters:
        - init_latents_w: The initial latents with watermarking.
        - w_mask_shape: The shape of the watermarking mask (e.g., "circle", "square").
        - w_channel: The channel to apply the watermarking mask to (-1 for all channels).
        - w_radius: The radius for the watermarking mask.
        - device: The device to use for computation (e.g., "cuda" or "cpu").
    Returns:
        - watermarking_mask: The generated watermarking mask.
    """
    watermarking_mask = torch.zeros(init_latents_w.shape, dtype=torch.bool).to(device)

    if w_mask_shape == "circle":
        np_mask = circle_mask(init_latents_w.shape[-1], r=w_radius)
        torch_mask = torch.tensor(np_mask).to(device)

        if w_channel == -1:
            # all channels
            watermarking_mask[:, :] = torch_mask
        else:
            watermarking_mask[:, w_channel] = torch_mask
    elif w_mask_shape == "square":
        anchor_p = init_latents_w.shape[-1] // 2
        if w_channel == -1:
            # all channels
            watermarking_mask[
                :,
                :,
                anchor_p - w_radius : anchor_p + w_radius,
                anchor_p - w_radius : anchor_p + w_radius,
            ] = True
        else:
            watermarking_mask[
                :,
                w_channel,
                anchor_p - w_radius : anchor_p + w_radius,
                anchor_p - w_radius : anchor_p + w_radius,
            ] = True
    elif w_mask_shape == "no":
        pass
    else:
        raise NotImplementedError(f"w_mask_shape: {w_mask_shape}")

    return watermarking_mask


init_latents_w = pipe.get_random_latents()
watermarking_mask = get_watermarking_mask(
        init_latents_w,
        w_mask_shape=w_mask_shape,
        w_channel=w_channel,
        w_radius=w_radius,
        device=device,
    )


In [9]:
def psnr_to_prob_sigmoid(psnr, threshold=-4.0, scale=1.0):
    """Simple sigmoid mapping. threshold -> p=0.5, smaller scale -> steeper curve."""
    return 1.0 / (1.0 + math.exp(-(psnr - threshold) / scale))

def detector(x):
    result = detect_watermark_ncx2_from_latent(
        latent_tensor=x,
        gt_patch=gt_patch,
        watermarking_mask=watermarking_mask,
        k_in_freq_domain=True,
        alpha=0.01,
        verbose=True,
    )

    # detected = result["detected"]
    # detected = bool(result['psnr_metric'] > -4.0)
    detected = psnr_to_prob_sigmoid(result['psnr_metric'], threshold=-4.0, scale=1.0)
    # turn into probability-like output
    return detected

def batch_detector(x_batch):
    results = []
    for i in range(x_batch.shape[0]):
        res = detect_watermark_ncx2_from_latent(
            latent_tensor=x_batch[i : i + 1],
            gt_patch=gt_patch,
            watermarking_mask=watermarking_mask,
            k_in_freq_domain=True,
            alpha=0.01,
            verbose=False,
        )
        detected = psnr_to_prob_sigmoid(res['psnr_metric'], threshold=-4.0, scale=1.0)
        results.append(float(detected))
        # results.append(float(res["detected"]))
    return torch.tensor(results, device=x_batch.device)

def eval_watermark_detector(detector, loader):
    trues, preds, probs = [], [], []
    with torch.no_grad():
        for X, y in tqdm(loader, desc="eval", leave=False):
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            out = detector(X)
            pred = (out > 0.5).cpu().numpy().astype(int).tolist()
            probs.extend(pred)
            preds.extend(pred)
            trues.extend(y.cpu().numpy().tolist())
    acc = accuracy_score(trues, preds)
    try:
        auc = roc_auc_score(trues, probs)
    except Exception:
        auc = float("nan")
    return acc, auc

In [10]:
all_accs, all_aucs = [], []
val_loader.dataset.set_return_reversed_latents(True)
for e_i in range(evaluate_times):
    acc, auc  = eval_watermark_detector(batch_detector, val_loader)
    all_accs.append(acc)
    all_aucs.append(auc)
    print(f"Val acc: {acc:.4f}, Val AUROC: {auc:.4f}")
print("Training complete.")
# On Augmented dataset, the performance is:
print(f"ACC [{np.array(all_accs).mean():.4f}+-{np.array(all_accs).std():.4f}] | AUROC [{np.array(all_aucs).mean():.4f}+-{np.array(all_aucs).std():.4f}] |")

Val acc: 0.7133, Val AUROC: 0.7271


Val acc: 0.6133, Val AUROC: 0.6308


Val acc: 0.6667, Val AUROC: 0.6821


Val acc: 0.6800, Val AUROC: 0.6941


Val acc: 0.6933, Val AUROC: 0.7074
Training complete.
ACC [0.6733+-0.0337] | AUROC [0.6883+-0.0324] |


In [11]:
# ACC [0.4733+-0.0000] | AUROC [0.5000+-0.0000] |

In [12]:
all_accs, all_aucs = [], []
val_loader_no_aug.dataset.set_return_reversed_latents(True)
for e_i in range(1):
    acc, auc  = eval_watermark_detector(batch_detector, val_loader_no_aug)
    all_accs.append(acc)
    all_aucs.append(auc)
    print(f"  val acc: {acc:.4f}  val AUROC: {auc:.4f}")
print("Training complete.")
# On Augmented dataset, the performance is:
print(f"ACC [{np.array(all_accs).mean():.4f}+-{np.array(all_accs).std():.4f}] | AUROC [{np.array(all_aucs).mean():.4f}+-{np.array(all_aucs).std():.4f}] |")

  val acc: 0.9400  val AUROC: 0.9395
Training complete.
ACC [0.9400+-0.0000] | AUROC [0.9395+-0.0000] |


In [13]:
# ACC [0.4733+-0.0000] | AUROC [0.5000+-0.0000] |

In [14]:
# val acc: 0.4733  val AUROC: 0.5000
# val acc: 0.4733  val AUROC: 0.5000